# Decision Trees: Learning by Asking Questions

**Session 1:** how does linear regression find the best line?
**Session 2:** how do we know whether that line is actually good, and whether individual variables matter?
**Session 3:** what happens when the thing we want to predict is a category instead of a number?
**This session:** what happens when a straight boundary line still isn't enough to separate the categories?

Let's find out.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split

pd.set_option("display.precision", 3)

We left Session 3 with a straight decision boundary separating pass from fail. Let's check how well that boundary handles a trickier version of the passing rule.

<details>
<summary>Show code</summary>

```python
pass_df = pd.read_csv("../data/exam_pass_3d.csv")

recap_model = LogisticRegression()
recap_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])

recap_accuracy = recap_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
baseline_accuracy = 1 - pass_df["passed"].mean()

print(f"Logistic regression accuracy:          {recap_accuracy:.3f}")
print(f"Baseline (always guess the majority):  {baseline_accuracy:.3f}")
```

</details>

In [2]:
pass_df = pd.read_csv("../data/exam_pass_3d.csv")

recap_model = LogisticRegression()
recap_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])

recap_accuracy = recap_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
baseline_accuracy = 1 - pass_df["passed"].mean()

print(f"Logistic regression accuracy:          {recap_accuracy:.3f}")
print(f"Baseline (always guess the majority):  {baseline_accuracy:.3f}")

Logistic regression accuracy:          0.764
Baseline (always guess the majority):  0.727


Barely better than just guessing the majority class every time. Something about this data is fighting the model.

## Why Would a Straight Boundary Fail Here?

The true rule behind `exam_pass_3d.csv` isn't "more hours always help" or "more practice always helps." It's a **sweet spot**: you need to study *enough*, and you need to practice a *moderate* amount -- too few practice problems and you're underprepared, too many and it starts to look like guessing/rushing.

A straight decision boundary can express "more of this feature always pushes toward pass" or "always pushes toward fail." It cannot express "there's a sweet spot in the middle" -- that shape needs at least one bend, and a straight line has none.

> ### 🙋 Ask the class
>
> - Can you think of other real-world rules that have a "too little OR too much" shape, where a straight-line rule wouldn't work?

## Thinking Like a Tree: The Twenty Questions Game

Before fitting anything, let's reason it out by hand -- the way you might play Twenty Questions.

1. **Did the student study at least 4 hours?** If no → predict **fail**.
2. If yes: **did they solve between 5 and 15 practice problems?** If yes → predict **pass**. If no → predict **fail**.

That's it. Two yes/no questions, asked in sequence, correctly separate most of the students in this dataset.

This is exactly what a decision tree does, just with vocabulary attached:

- **Root** — the first question asked of every row (`hours_studied >= 4?`)
- **Split** — any yes/no question that divides the data into two groups
- **Leaf** — a final box with no more questions, just a prediction

> ### 🙋 Ask the class
>
> - If you're playing a number-guessing game (1-100) and can only ask yes/no questions, what's the fewest questions you need in the worst case? How does that connect to how deep a tree needs to be?

## Anatomy of a Tree

```text
                     hours_studied >= 4?
                    /                   \
                  no                    yes
                   |                     |
                 FAIL          practice_problems in [5, 15]?
                                /                        \
                              yes                        no
                               |                          |
                             PASS                       FAIL
```

The top question is the **root**. Every question below it is a **split**. Every box with a final answer is a **leaf**.

## From Boundary Line to Boundary Staircase

Let's see if a tree actually does better. First, a quick check against data you already know.

<details>
<summary>Show code</summary>

```python
warmup_df = pd.read_csv("../data/exam_pass_2d.csv")

warmup_logistic = LogisticRegression()
warmup_logistic.fit(warmup_df[["hours_studied", "practice_problems"]], warmup_df["passed"])

warmup_tree = DecisionTreeClassifier(max_depth=3, random_state=0)
warmup_tree.fit(warmup_df[["hours_studied", "practice_problems"]], warmup_df["passed"])

print(f"Logistic regression accuracy (Session 3 data): {warmup_logistic.score(warmup_df[['hours_studied', 'practice_problems']], warmup_df['passed']):.3f}")
print(f"Decision tree accuracy (Session 3 data):        {warmup_tree.score(warmup_df[['hours_studied', 'practice_problems']], warmup_df['passed']):.3f}")
```

</details>

In [3]:
warmup_df = pd.read_csv("../data/exam_pass_2d.csv")

warmup_logistic = LogisticRegression()
warmup_logistic.fit(warmup_df[["hours_studied", "practice_problems"]], warmup_df["passed"])

warmup_tree = DecisionTreeClassifier(max_depth=3, random_state=0)
warmup_tree.fit(warmup_df[["hours_studied", "practice_problems"]], warmup_df["passed"])

print(f"Logistic regression accuracy (Session 3 data): {warmup_logistic.score(warmup_df[['hours_studied', 'practice_problems']], warmup_df['passed']):.3f}")
print(f"Decision tree accuracy (Session 3 data):        {warmup_tree.score(warmup_df[['hours_studied', 'practice_problems']], warmup_df['passed']):.3f}")

Logistic regression accuracy (Session 3 data): 0.820
Decision tree accuracy (Session 3 data):        0.880


Not a huge difference on that older dataset -- because that data's true boundary really was close to a straight line. Let's look at data where it isn't.

<details>
<summary>Show code</summary>

```python
tree_2feat_model = DecisionTreeClassifier(max_depth=3, random_state=0)
tree_2feat_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])

logistic_2feat_model = LogisticRegression()
logistic_2feat_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])

tree_accuracy = tree_2feat_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
logistic_accuracy = logistic_2feat_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
print(f"Logistic regression accuracy: {logistic_accuracy:.3f}")
print(f"Decision tree accuracy:       {tree_accuracy:.3f}")

grid_hours = np.linspace(pass_df["hours_studied"].min(), pass_df["hours_studied"].max(), 200)
grid_practice = np.linspace(pass_df["practice_problems"].min(), pass_df["practice_problems"].max(), 200)
grid_x, grid_y = np.meshgrid(grid_hours, grid_practice)
grid_points = pd.DataFrame({"hours_studied": grid_x.ravel(), "practice_problems": grid_y.ravel()})
grid_predictions = tree_2feat_model.predict(grid_points).reshape(grid_x.shape)

w1, w2 = logistic_2feat_model.coef_[0]
b = logistic_2feat_model.intercept_[0]
boundary_x1 = np.linspace(pass_df["hours_studied"].min(), pass_df["hours_studied"].max(), 50)
boundary_x2 = -(b + w1 * boundary_x1) / w2

boundary_figure = go.Figure()
boundary_figure.add_trace(
    go.Contour(
        x=grid_hours, y=grid_practice, z=grid_predictions,
        showscale=False, colorscale=[[0, "rgba(220,38,38,0.15)"], [1, "rgba(22,163,74,0.15)"]],
        contours=dict(start=0, end=1, size=1, coloring="fill"),
        line=dict(width=0), hoverinfo="skip",
    )
)
for passed_value, color, label in [(0, "#dc2626", "actual: failed"), (1, "#16a34a", "actual: passed")]:
    subset = pass_df[pass_df["passed"] == passed_value]
    boundary_figure.add_trace(
        go.Scatter(
            x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
            marker=dict(size=9, color=color, line=dict(width=1, color="white")), name=label,
        )
    )
boundary_figure.add_trace(
    go.Scatter(
        x=boundary_x1, y=boundary_x2, mode="lines",
        line=dict(color="#7c3aed", width=3, dash="dash"), name="logistic regression boundary",
    )
)
boundary_figure.update_layout(
    title="Decision Tree Region vs. Logistic Regression Boundary",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
boundary_figure.show()
```

</details>

In [4]:
tree_2feat_model = DecisionTreeClassifier(max_depth=3, random_state=0)
tree_2feat_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])

logistic_2feat_model = LogisticRegression()
logistic_2feat_model.fit(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])

tree_accuracy = tree_2feat_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
logistic_accuracy = logistic_2feat_model.score(pass_df[["hours_studied", "practice_problems"]], pass_df["passed"])
print(f"Logistic regression accuracy: {logistic_accuracy:.3f}")
print(f"Decision tree accuracy:       {tree_accuracy:.3f}")

grid_hours = np.linspace(pass_df["hours_studied"].min(), pass_df["hours_studied"].max(), 200)
grid_practice = np.linspace(pass_df["practice_problems"].min(), pass_df["practice_problems"].max(), 200)
grid_x, grid_y = np.meshgrid(grid_hours, grid_practice)
grid_points = pd.DataFrame({"hours_studied": grid_x.ravel(), "practice_problems": grid_y.ravel()})
grid_predictions = tree_2feat_model.predict(grid_points).reshape(grid_x.shape)

w1, w2 = logistic_2feat_model.coef_[0]
b = logistic_2feat_model.intercept_[0]
boundary_x1 = np.linspace(pass_df["hours_studied"].min(), pass_df["hours_studied"].max(), 50)
boundary_x2 = -(b + w1 * boundary_x1) / w2

boundary_figure = go.Figure()
boundary_figure.add_trace(
    go.Contour(
        x=grid_hours, y=grid_practice, z=grid_predictions,
        showscale=False, colorscale=[[0, "rgba(220,38,38,0.15)"], [1, "rgba(22,163,74,0.15)"]],
        contours=dict(start=0, end=1, size=1, coloring="fill"),
        line=dict(width=0), hoverinfo="skip",
    )
)
for passed_value, color, label in [(0, "#dc2626", "actual: failed"), (1, "#16a34a", "actual: passed")]:
    subset = pass_df[pass_df["passed"] == passed_value]
    boundary_figure.add_trace(
        go.Scatter(
            x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
            marker=dict(size=9, color=color, line=dict(width=1, color="white")), name=label,
        )
    )
boundary_figure.add_trace(
    go.Scatter(
        x=boundary_x1, y=boundary_x2, mode="lines",
        line=dict(color="#7c3aed", width=3, dash="dash"), name="logistic regression boundary",
    )
)
boundary_figure.update_layout(
    title="Decision Tree Region vs. Logistic Regression Boundary",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
boundary_figure.show()

Logistic regression accuracy: 0.764
Decision tree accuracy:       0.982


The tree's shaded region wraps around the "sweet spot" pass zone -- something a single straight line geometrically cannot do. That's the gap between roughly 0.76 and 0.98 accuracy you just saw printed above.

## Training Data vs. New Data: A Quick Detour

So far we've graded every model on the same data it learned from. That's fine for a straight line -- it only has
so much flexibility to begin with. But trees are flexible enough to potentially just *memorize* the training rows
rather than learn the real pattern.

To catch that, we hold out a slice of data the model never sees while training, and grade it only on that slice.

<details>
<summary>Show code</summary>

```python
X = pass_df[["hours_studied", "practice_problems"]]
y = pass_df["passed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")
```

</details>

In [5]:
X = pass_df[["hours_studied", "practice_problems"]]
y = pass_df["passed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")

Training rows: 77
Test rows:     33


## Tree Depth: Underfitting, Overfitting, and the Sweet Spot

`max_depth` limits how many questions a tree is allowed to ask before it must land on a leaf. Let's compare a
shallow tree, a moderate one, and one with no limit at all.

<details>
<summary>Show code</summary>

```python
depth_results = []
for depth in [1, 3, None]:
    depth_model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    depth_model.fit(X_train, y_train)
    depth_results.append({
        "max_depth": depth if depth is not None else "None (unlimited)",
        "train_accuracy": depth_model.score(X_train, y_train),
        "test_accuracy": depth_model.score(X_test, y_test),
    })

pd.DataFrame(depth_results)
```

</details>

In [6]:
depth_results = []
for depth in [1, 3, None]:
    depth_model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    depth_model.fit(X_train, y_train)
    depth_results.append({
        "max_depth": depth if depth is not None else "None (unlimited)",
        "train_accuracy": depth_model.score(X_train, y_train),
        "test_accuracy": depth_model.score(X_test, y_test),
    })

pd.DataFrame(depth_results)

,max_depth,train_accuracy,test_accuracy
0,1,0.779,0.697
1,3,0.974,0.939
2,None (unlimited),1.000,0.879


> ### 🧑‍🏫 Instructor note
>
> "More depth" feels like it should mean "more accurate" -- and on the TRAIN column it does, right up to a
> perfect 1.000. But look at the TEST column: the depth=3 tree beats the unlimited-depth tree there. The
> unlimited tree memorized quirks of the training rows that don't generalize to new students -- that's overfitting.

## Reading the Tree Itself

No fancy diagram needed for this -- scikit-learn can print the tree's actual questions as text.

<details>
<summary>Show code</summary>

```python
depth3_model = DecisionTreeClassifier(max_depth=3, random_state=0)
depth3_model.fit(X_train, y_train)
print(export_text(depth3_model, feature_names=["hours_studied", "practice_problems"]))
```

</details>

In [7]:
depth3_model = DecisionTreeClassifier(max_depth=3, random_state=0)
depth3_model.fit(X_train, y_train)
print(export_text(depth3_model, feature_names=["hours_studied", "practice_problems"]))

|--- hours_studied <= 5.45
|   |--- hours_studied <= 4.20
|   |   |--- class: 0
|   |--- hours_studied >  4.20
|   |   |--- hours_studied <= 4.45
|   |   |   |--- class: 1
|   |   |--- hours_studied >  4.45
|   |   |   |--- class: 0
|--- hours_studied >  5.45
|   |--- practice_problems <= 4.50
|   |   |--- class: 0
|   |--- practice_problems >  4.50
|   |   |--- practice_problems <= 15.50
|   |   |   |--- class: 1
|   |   |--- practice_problems >  15.50
|   |   |   |--- class: 0



Compare this to the two questions we wrote by hand earlier -- the fitted tree found nearly the same rule on its own, just from the data.

## What a Split Is Actually Optimizing

$$\text{Gini} = 1 - \sum_i p_i^2$$

You don't need to memorize this. Just like Session 1's SSE, it's simply the number the algorithm is trying to
minimize every time it chooses a split: lower Gini after a split means the two resulting groups are more "pure"
(mostly one class). The tree tries every possible split on every feature and picks whichever one drops Gini the most.

## Feature Importance: A Preview of Session 5

`exam_pass_3d.csv` actually has a third column, `sleep_hours`, that we haven't used yet -- and it's deliberately
unrelated to whether a student passes. Let's see if the tree notices.

<details>
<summary>Show code</summary>

```python
X3 = pass_df[["hours_studied", "practice_problems", "sleep_hours"]]
y3 = pass_df["passed"]

importance_model = DecisionTreeClassifier(max_depth=3, random_state=0)
importance_model.fit(X3, y3)

importance_figure = go.Figure()
importance_figure.add_trace(
    go.Bar(x=list(X3.columns), y=importance_model.feature_importances_, marker=dict(color="#f59e0b"))
)
importance_figure.update_layout(
    title="Feature Importance (single tree)",
    xaxis_title="feature", yaxis_title="importance",
    template="plotly_white", width=650, height=450,
)
importance_figure.show()
```

</details>

In [8]:
X3 = pass_df[["hours_studied", "practice_problems", "sleep_hours"]]
y3 = pass_df["passed"]

importance_model = DecisionTreeClassifier(max_depth=3, random_state=0)
importance_model.fit(X3, y3)

importance_figure = go.Figure()
importance_figure.add_trace(
    go.Bar(x=list(X3.columns), y=importance_model.feature_importances_, marker=dict(color="#f59e0b"))
)
importance_figure.update_layout(
    title="Feature Importance (single tree)",
    xaxis_title="feature", yaxis_title="importance",
    template="plotly_white", width=650, height=450,
)
importance_figure.show()

`sleep_hours` lands at essentially zero importance -- the tree never found a split on it worth making. This is the tree's own version of Session 2's p-values: a way of unmasking a feature that looks like data but carries no signal.

## Recap

```text
data
  ↓
try every possible split on every feature, pick the one that best separates the classes (lowest Gini)
  ↓
repeat inside each resulting group
  ↓
stop (leaf) when a group is pure enough, or max_depth is reached
  ↓
prediction = majority class in that leaf
```

**A single tree can carve out almost any shape a straight boundary can't -- but give it enough depth and it will
carve out the training data's noise just as eagerly as its signal.**

> One tree can fit almost anything, which is exactly the problem. What if, instead of trusting one tree's
> judgment call, we asked many different trees and combined their answers?